# Module 08 · Voltage regulators (Floyd Ch.17)

In [1]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Line and load regulation

In [2]:
print(f"Prob 1: line = {2e-3/6*100:.4f} %  ({2e-3/6*100/6:.5f} %/V)")
print(f"Prob 3: load = {(10-9.90)/9.90*100:.2f} %  ({(10-9.90)/9.90*100/250:.5f} %/mA)")

Prob 1: line = 0.0333 %  (0.00556 %/V)
Prob 3: load = 1.01 %  (0.00404 %/mA)


**Comment:** 0.0333 % and 1.01 % match the book.

## 2. Series regulator with op-amp: VIN changes, VOUT holds

In [3]:
def series(vin, vref=5.1, r2=10e3, r3=10e3, A=1e5, vdrop=0.7):
    # error-amplifier loop solved by iteration: Vout = A*(vref - Vout*r3/(r2+r3)) limited by VIN - vdrop
    b = r3/(r2+r3); vo = vref/b
    vo_open = min(A*vref/(1+A*b), vin - vdrop)
    return vo_open
for vin in (13, 15, 16, 20): print(f"VIN = {vin} V -> VOUT = {series(vin):.3f} V")
print("ideal (1+R2/R3)*VREF =", (1+1)*5.1)

VIN = 13 V -> VOUT = 10.200 V
VIN = 15 V -> VOUT = 10.200 V
VIN = 16 V -> VOUT = 10.200 V
VIN = 20 V -> VOUT = 10.200 V
ideal (1+R2/R3)*VREF = 10.2


**Comment:** for VIN = 13, 15, 16 and 20 V the output is 10.200 V = (1 + R2/R3)·VREF; the error from the op-amp’s finite gain is below 1 mV so it does not show at three decimals. The output would only fall if VIN dropped below VOUT + 0.7 V (out of headroom, in this simple model). Problems 7–9 give 8.51 V, 5.26 V (R3 = 4.7 kΩ) and 9.57 V.

## 3. Current limiting and the shunt regulator

In [4]:
print("Prob 10: R4 =", 0.7/0.25, "ohm, P =", round(0.25**2*0.7/0.25, 3), "W; halving R4 ->", 0.7/1.4*1e3, "mA")
print("Prob 13: dIC =", 1/100*1e3, "mA;  Prob 15: IL(max) =", 25/100*1e3, "mA, PR1 =", 25**2/100, "W")
vin = np.linspace(12, 25, 6)
print("shunt: R1 current with Vout = 10 V:", np.round((vin-10)/100*1e3, 1), "mA")

Prob 10: R4 = 2.8 ohm, P = 0.175 W; halving R4 -> 500.0 mA
Prob 13: dIC = 10.0 mA;  Prob 15: IL(max) = 250.0 mA, PR1 = 6.25 W
shunt: R1 current with Vout = 10 V: [ 20.  46.  72.  98. 124. 150.] mA


**Comment:** R4 = 2.8 Ω, 0.175 W; halving R4 gives 500 mA; shunt regulator: 10 mA per volt of VIN, 250 mA and 6.25 W when shorted — all as in the book.

## 4. Switching regulator: duty cycle and LC filtering

In [5]:
vin, f, d, L, C, R = 12.0, 10e3, 0.4, 1e-3, 100e-6, 10.0
dt = 1e-7; n = int(0.02/dt); i = 0.0; v = 0.0; vs = []
for k in range(n):
    t = (k*dt) % (1/f); sw = vin if t < d/f else 0.0
    i += (sw - v)/L*dt; v += (i - v/R)/C*dt; vs.append(v)
vs = np.array(vs); tail = vs[-int(0.002/dt):]
print(f"duty = {d*100:.0f} %: ideal Vout = {d*vin:.2f} V; simulated mean = {tail.mean():.2f} V, ripple = {tail.max()-tail.min():.3f} V")
print("Prob 17: duty =", (1000/10-60)/(1000/10)*100, "%")

duty = 40 %: ideal Vout = 4.80 V; simulated mean = 4.80 V, ripple = 0.040 V
Prob 17: duty = 40.0 %


**Comment:** the LC simulation at a 40 % duty cycle settles near 4.8 V = 0.4 × 12 V with small ripple; Problem 17 gives 40 % as in the book.

## 5. LM317 and 78XX ICs

In [6]:
v = 1.25*(1+10/1) + 50e-6*10e3; print(f"Prob 21: VOUT = {v:.2f} V;  Prob 23: I = {v/11e3*1e3:.2f} mA")
print("Prob 25: Rext =", 0.7/0.25, "ohm;  Prob 27: Rlimit =", 0.7/2, "ohm")

Prob 21: VOUT = 14.25 V;  Prob 23: I = 1.30 mA
Prob 25: Rext = 2.8 ohm;  Prob 27: Rlimit = 0.35 ohm


**Comment:** 14.25 V ≈ 14.3 V, 1.3 mA, 2.8 Ω and 0.35 Ω all match the book.